# 01. Build the RUL dataset from the raw ISU-ILCC records

Turns the raw cycling records of the ISU-ILCC Battery Aging Dataset (about 11 GB) into one small table, `rul_steps.csv`, with one row per weekly capacity test of every cell. The table is already included in `data/`, so this notebook only needs to be run to reproduce it.

Run in Google Colab with `RPT_json.zip`, `Cycling_json.zip` and `Valid_cells.csv` in the top level of Google Drive.

## Setup
Mount Google Drive and set paths and constants.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, glob, json, gc, zipfile
import numpy as np
import pandas as pd

ROOT = '/content/drive/MyDrive'
VALID_CELLS = f'{ROOT}/Valid_cells.csv'
OUTPUT = f'{ROOT}/rul_steps.csv'

RATED_AH = 0.250           # rated capacity
EOL_AH = 0.8 * RATED_AH    # end of life: 80% of rated capacity

## Reading the raw files
Each cell has an RPT file (weekly capacity tests) and a cycling file (every charge-discharge cycle). Both are JSON wrapped in a JSON string, and missing records are stored as `'[]'`.

In [ ]:
def json_paths(name):
    # Cell name -> JSON path. A ZIP, if present, is extracted inside Colab (faster than reading from Drive).
    zip_path, out = f'{ROOT}/{name}.zip', f'/content/data/{name}'
    if not os.path.exists(out) and os.path.exists(zip_path):
        print(f'Extracting {name}.zip ...')
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(out)
    for base in [out, f'{ROOT}/{name}']:
        paths = sorted(glob.glob(f'{base}/**/*.json', recursive=True))
        if paths:
            return {os.path.splitext(os.path.basename(p))[0]: p for p in paths}
    return {}

def load_json(path):
    with open(path) as f:
        return json.loads(json.load(f))   # decode twice

def to_time(values):
    clean = pd.Series([v if isinstance(v, str) and v != '[]' else None for v in values], dtype=object)
    return pd.to_datetime(clean, format='ISO8601', errors='coerce')

def to_num(values):
    return pd.Series([v if isinstance(v, (int, float)) else np.nan for v in values], dtype=float)

def first_currents(cyc, n_cycles=30):
    # (max charge current, median discharge current) over the first n_cycles cycles
    chg, dis = [], []
    for i_chg, i_dis in zip(cyc['QV_charge']['I'][:n_cycles], cyc['QV_discharge']['I'][:n_cycles]):
        if len(i_chg):
            chg.append(np.max(np.abs(i_chg)))
        if len(i_dis):
            dis.append(np.median(np.abs(i_dis)))
    return (np.median(chg) if chg else np.nan), (np.median(dis) if dis else np.nan)

## One row per weekly test
For every weekly test before the end of life:

- `capacity`: full capacity measured at that test (Ah)
- `efc_so_far`: equivalent full cycles (EFC) used before that test = cumulative discharged Ah / 0.25
- `rul_efc`: EFC remaining until capacity reaches 80% of rated capacity (the prediction target)
- `c_chg`, `c_dis`, `dod`: charge rate, discharge rate and depth of discharge, measured from the first cycles

The end of life is interpolated between the two weekly tests around the 80% threshold.

In [ ]:
def rul_rows(rpt, cyc):
    # Weekly tests: capacity and start time (keep tests that have both)
    cap, t_rpt = to_num(rpt['capacity_discharge_C_5']), to_time(rpt['start_stop_time']['start'])
    n = min(len(cap), len(t_rpt))
    ok = (cap[:n].notna() & t_rpt[:n].notna()).values
    cap, t_rpt = cap[:n].values[ok], t_rpt[:n].values[ok]

    # Cycles: discharged capacity and discharge start time, sorted by time
    q_all, t_cyc = to_num(cyc['capacity_discharge']), to_time(cyc['time_series_discharge']['start'])
    n = min(len(q_all), len(t_cyc))
    # A missing capacity is filled with the median of the cell's earlier cycles only, so no later
    # information enters the EFC used so far (the counts printed below show none were missing).
    q = q_all[:n]
    q, t_cyc = q.fillna(q.expanding().median().shift(1)).fillna(q.dropna().iloc[:10].median()), t_cyc[:n]
    keep = t_cyc.notna().values
    q, t_cyc = q.values[keep], t_cyc.values[keep]
    order = np.argsort(t_cyc)
    q, t_cyc = q[order], t_cyc[order]

    n_before = np.searchsorted(t_cyc, t_rpt)                                 # cycles started before each test
    efc_before = np.concatenate([[0.0], np.cumsum(q)])[n_before] / RATED_AH  # EFC used before each test

    below = np.where(cap < EOL_AH)[0]
    if not len(q) or not len(below) or below[0] == 0:
        return []                                  # never reached end of life, or no usable record
    i = below[0]
    frac = (cap[i-1] - EOL_AH) / (cap[i-1] - cap[i])
    efc_total = efc_before[i-1] + frac * (efc_before[i] - efc_before[i-1])   # EFC at the 80% crossing

    i_chg, i_dis = first_currents(cyc)
    first_q = q_all.dropna().values[:10]
    conditions = {'c_chg': i_chg / RATED_AH, 'c_dis': i_dis / RATED_AH,
                  'dod': float(np.median(first_q) / cap[0])}
    return [{'step': k, 'capacity': float(cap[k]), 'efc_so_far': float(efc_before[k]),
             'rul_efc': float(efc_total - efc_before[k]), 'efc_total': float(efc_total), **conditions}
            for k in range(i)]                     # tests before the end of life

valid = pd.read_csv(VALID_CELLS, header=None, dtype=str).values.ravel()
valid = [v.strip() for v in valid if isinstance(v, str) and re.fullmatch(r'G\d+C\d+', v.strip())]
rpt_paths, cyc_paths = json_paths('RPT_json'), json_paths('Cycling_json')
todo = [c for c in valid if c in rpt_paths and c in cyc_paths]
print('Cells to read:', len(todo))

rows = []
n_filled = n_no_time = 0
for n_done, cell in enumerate(todo, 1):
    rpt, cyc = load_json(rpt_paths[cell]), load_json(cyc_paths[cell])
    q_raw, t_raw = to_num(cyc['capacity_discharge']), to_time(cyc['time_series_discharge']['start'])
    n = min(len(q_raw), len(t_raw))
    n_filled += int((q_raw[:n].isna() & t_raw[:n].notna()).sum())       # capacities rul_rows has to fill
    n_no_time += int(t_raw[:n].isna().sum())                             # cycles left out of the EFC count
    group = int(re.fullmatch(r'G(\d+)C(\d+)', cell).group(1))     # cells in a group share the same conditions
    rows += [{'cell': cell, 'group': group, **r} for r in rul_rows(rpt, cyc)]
    del rpt, cyc
    gc.collect()
    if n_done % 25 == 0:
        print(f'  {n_done}/{len(todo)} done')

print('Cycles with a missing capacity (filled):', n_filled, '| cycles without a start time (not counted):', n_no_time)
steps = pd.DataFrame(rows)
steps.to_csv(OUTPUT, index=False)
print('Saved', OUTPUT, '|', steps['cell'].nunique(), 'cells,', len(steps), 'rows')